In [ ]:
# Display the output of all lines in a cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
from pprint import pprint

import torch
from huggingface_hub import HfApi

import pandas as pd

import lerobot
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

In [ ]:
# Browse through the datasets created/ported by the community on the hub using the hub api:
hub_api = HfApi()
repo_ids = [info.id for info in hub_api.list_datasets(task_categories="robotics", tags=["LeRobot"])]

In [ ]:
# Filter to find datasets related to physical intelligence
#[s for s in repo_ids if ("lerobot" in s.lower() and "so101" in s.lower())]
#[s for s in repo_ids if ("libero" in s.lower())]
[s for s in repo_ids if ("aloha_mobile" in s.lower())]



In [ ]:
# Load libero
repo_id= "lerobot/svla_so101_pickplace"
repo_id = "lerobot/droid_100"
repo_id = "lerobot/aloha_mobile_wash_pan"
repo_id = "physical-intelligence/libero"

# We can have a look and fetch its metadata to know more about it:
ds_meta = LeRobotDatasetMetadata(repo_id)

In [ ]:
print(f"Total number of episodes: {ds_meta.total_episodes}")
print(f"Average number of frames per episode: {ds_meta.total_frames / ds_meta.total_episodes:.3f}")
print(f"Frames per second used during data collection: {ds_meta.fps}")
print(f"Robot type: {ds_meta.robot_type}")
print(f"keys to access images from cameras: {ds_meta.camera_keys=}\n")

print("Tasks:")
print(ds_meta.tasks)
print("Features:")
pprint(ds_meta.features)

# You can also get a short summary by simply printing the object:
print(ds_meta)

In [ ]:
dataset = LeRobotDataset(repo_id)

In [ ]:
dataset

In [ ]:

frames = []
for item in dataset:
    frame = {
        "state": item["state"].tolist() if isinstance(item["state"], torch.Tensor) else item["state"],
        "actions": [a.tolist() if isinstance(a, torch.Tensor) else a for a in item["actions"]],
        "timestamp": item["timestamp"].item() if isinstance(item["timestamp"],torch.Tensor) else item["timestamp"],
        "frame_index": item["frame_index"].item() if isinstance(item["frame_index"], torch.Tensor) else item["frame_index"],
        "episode_index": item["episode_index"].item() if isinstance(item["episode_index"], torch.Tensor) else item["episode_index"],
        "index": item["index"].item() if isinstance(item["index"], torch.Tensor) else item["index"],
        "task_index": item["task_index"].item() if isinstance(item["task_index"], torch.Tensor) else item["task_index"],
        "task": item["task"],  # assuming this is already a string or int
    }
    frames.append(frame)



In [ ]:
df = pd.DataFrame(frames)


In [ ]:
df

In [ ]:
action_col_names = ["action_" + str(i) for i in range(len(df.actions[0]))]
state_col_names = ["state_" + str(i) for i in range(len(df.state[0]))]
df[state_col_names] = pd.DataFrame(df.state.tolist(), index= df.index)
df[action_col_names] = pd.DataFrame(df.actions.tolist(), index= df.index)

In [ ]:
df[df["task_index"] == 34].describe()

In [ ]:
for item in ds_meta.tasks.items():
    print(f"Task {item[0]}: {item[1]}")
    